# AgenticRAG-Bench — Week 5
## Hybrid BM25+FAISS Retrieval + Rewrite Validation

**What's new in Week 5:**
1. Hybrid retrieval — BM25 (keyword) + FAISS (dense) with reciprocal rank fusion
2. Rewrite validation — detect and reject bad LLM rewrites (meta-language fallback)
3. Ablation: run 3 configurations to isolate each lever's contribution

**What does NOT change from Week 4:**
- Agent: LangGraph ReAct + Llama 3.1 8B
- Embeddings: nomic-embed-text (for FAISS component)
- k = 3 | system prompt ON | max_steps = 10

**Week 4 baseline (what we're trying to beat):**
- D1 = 30% (15/50 correct)
- D2 = 0.302 (15/48 questions D2=0)
- D3 = 0.647 (planning is flat — not the bottleneck)
- D4 = −0.133 (noise helped — constructive interference)

**Root cause from Week 4 analysis:**
- 14/15 D2=0 questions have word overlap between query and supporting paragraphs — the agent searches for the right thing, but FAISS can't find it among 18 distractors
- 15/100 search steps (15%) have bad rewrites with meta-language like "Similarity search for..."
- Embedding model is the bottleneck, not query formulation

**Ablation plan:**

| Config | Rewrite validation | BM25 hybrid | Purpose |
|---|---|---|---|
| A (baseline) | OFF | OFF | Week 4 reproduction — query rewriting only |
| B | ON | OFF | Isolate rewrite validation effect |
| C | ON | ON | Full pipeline — both levers |

---

## Cell 0 — Environment + imports

In [2]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rank_bm25"])

import sys
import os
import json
import time
import math
import re
import copy
import random
import signal
from pathlib import Path
from difflib import SequenceMatcher
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
os.environ["RAGAS_DO_NOT_PARALLELIZE"] = "true"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # Fix OMP libomp double-init crash with FAISS
load_dotenv(PROJECT_ROOT / ".env")

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version)

import requests
r = requests.get("http://localhost:11434/api/tags", timeout=3)
models = [m['name'] for m in r.json().get('models', [])]
print("Ollama models:", models)

from datasets import load_dataset
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.tools import tool
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langgraph.prebuilt import create_react_agent
from rank_bm25 import BM25Okapi
import numpy as np

Path(PROJECT_ROOT / "data/questions").mkdir(parents=True, exist_ok=True)
Path(PROJECT_ROOT / "data/trajectories").mkdir(parents=True, exist_ok=True)
Path(PROJECT_ROOT / "notes").mkdir(parents=True, exist_ok=True)

print("All imports OK (including rank_bm25)")

Project root: /Users/idhantsingh/Desktop/agenticrag-bench
Python: 3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 17.0.0 (clang-1700.6.3.2)]
Ollama models: ['nomic-embed-text:latest', 'llama3.1:8b']
All imports OK (including rank_bm25)



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


---
## Cell 1 — All metric classes (D1, D2, D3, D5)

Unchanged from Week 4.

In [3]:
# ── D1: Answer Correctness ───────────────────────────────────────

class D1_AnswerCorrectness:
    name = "D1_answer_correctness"
    STOPWORDS = {
        "the","a","an","is","was","of","in","to","and","or",
        "it","its","be","been","being","have","has","had",
        "by","at","from","with","that","this","are"
    }
    LEGAL_SUFFIXES = {
        "inc","incorporated","corp","corporation",
        "ltd","limited","llc","plc","gmbh","company"
    }

    def _normalize(self, text):
        text = text.lower().strip()
        text = re.sub(r'[^\w\s]', ' ', text)
        words = [w for w in text.split() if w not in self.LEGAL_SUFFIXES]
        return ' '.join(words)

    def _meaningful_words(self, text):
        text = re.sub(r'[^\w\s]', ' ', text)
        return {w for w in text.split()
                if w not in self.STOPWORDS
                and w not in self.LEGAL_SUFFIXES
                and len(w) > 1}

    def _is_degenerate(self, predicted):
        """Detect degenerate outputs: raw tool dumps, refusals, empty."""
        p = predicted.strip()
        return (
            (p.startswith('{') and 'search_knowledge_base' in p)
            or p.lower().startswith('sorry')
            or len(p) < 5
        )

    def score(self, predicted, ground_truth):
        degenerate = self._is_degenerate(predicted)
        if degenerate:
            return {
                "exact_match": False, "near_match": False,
                "token_recall": 0.0, "seq_sim": 0.0,
                "score": 0.0, "degenerate_output": True
            }

        pred_raw  = predicted.lower().strip()
        truth_raw = ground_truth.lower().strip()
        exact_raw  = truth_raw in pred_raw
        pred_norm  = self._normalize(predicted)
        truth_norm = self._normalize(ground_truth)
        if ' ' not in truth_norm:
            exact_norm = (
                pred_norm == truth_norm or
                pred_norm.startswith(truth_norm + ' ') or
                (' is ' + truth_norm) in pred_norm or
                (' was ' + truth_norm) in pred_norm
            )
        else:
            exact_norm = truth_norm in pred_norm
        exact_match = exact_raw or exact_norm
        pred_words  = self._meaningful_words(pred_norm)
        truth_words = self._meaningful_words(truth_norm)
        if truth_words:
            recall = len(pred_words & truth_words) / len(truth_words)
        else:
            recall = 1.0 if exact_match else 0.0
        seq = SequenceMatcher(None, truth_norm,
                              pred_norm[:len(truth_norm)*3]).ratio()
        near = (recall >= 0.6) or (seq >= 0.7 and recall >= 0.4)
        if exact_match:    score = 1.0
        elif near:         score = round(0.5 + recall * 0.5, 3)
        else:
            intersection = pred_words & truth_words
            union        = pred_words | truth_words
            score = round(len(intersection)/len(union), 3) if union else 0.0
        return {
            "exact_match": exact_match, "near_match": near,
            "token_recall": round(recall,3), "seq_sim": round(seq,3),
            "score": score, "degenerate_output": False
        }




# ── D2: Retrieval Step Quality ───────────────────────────────────

class D2_RetrievalStepQuality:
    name = "D2_retrieval_step_quality"
    def _doc_is_relevant(self, doc_text, supporting_paragraphs):
        doc_lower = doc_text.lower()
        for sup in supporting_paragraphs:
            if sup[:80].lower() in doc_lower:
                return True
            sup_words = set(sup.lower().split())
            doc_words = set(doc_lower.split())
            if sup_words and len(sup_words & doc_words) / len(sup_words) > 0.7:
                return True
        return False

    def score(self, trajectory_steps, supporting_paragraphs):
        if not trajectory_steps:
            return {"avg_precision_at_k":0.0, "best_step_precision":0.0,
                    "any_relevant_found":False, "step_precisions":[], "d2_score":0.0}
        step_precisions = []
        any_relevant = False
        for step in trajectory_steps:
            docs = step.get("retrieved_content", [])
            if not docs:
                step_precisions.append(0.0)
                continue
            hits = sum(1 for doc in docs if self._doc_is_relevant(doc, supporting_paragraphs))
            prec = hits / len(docs)
            step_precisions.append(round(prec, 3))
            if hits > 0: any_relevant = True
        avg_prec = sum(step_precisions) / len(step_precisions)
        best_prec = max(step_precisions)
        d2_score = round(min(1.0, avg_prec * 0.8 + (0.2 if any_relevant else 0.0)), 3)
        return {"avg_precision_at_k": round(avg_prec,3), "best_step_precision": round(best_prec,3),
                "any_relevant_found": any_relevant, "step_precisions": step_precisions, "d2_score": d2_score}




# ── D3: Planning Coherence ───────────────────────────────────────

class D3_PlanningCoherence:
    name = "D3_planning_coherence"
    def _query_diversity(self, queries):
        if len(queries) <= 1: return 0.0
        unique = list(set(queries))
        if len(unique) == 1: return 0.0
        pairs = []
        for i in range(len(unique)):
            for j in range(i+1, len(unique)):
                pairs.append(1.0 - SequenceMatcher(None, unique[i], unique[j]).ratio())
        return round(sum(pairs)/len(pairs), 3) if pairs else 0.0

    def score(self, trajectory_summary, num_hops):
        total_steps    = trajectory_summary.get("total_steps", 0)
        unique_queries = trajectory_summary.get("unique_queries", [])
        redundancy     = trajectory_summary.get("redundancy_rate", 0)
        loop_detected  = trajectory_summary.get("loop_detected", False)
        num_unique = len(unique_queries)
        hop_coverage = min(1.0, num_unique / max(num_hops, 1))
        diversity = self._query_diversity(unique_queries)
        loop_penalty      = 0.4 if loop_detected else 0.0
        undershoot        = max(0.0, (num_hops - num_unique) / max(num_hops, 1))
        undershoot_penalty = undershoot * 0.5
        raw = (hop_coverage * 0.5) + (diversity * 0.5) - loop_penalty - undershoot_penalty
        d3_score = round(max(0.0, min(1.0, raw)), 3)
        return {"num_hops_required": num_hops, "unique_queries_made": num_unique,
                "hop_coverage": round(hop_coverage,3), "query_diversity": round(diversity,3),
                "loop_penalty": loop_penalty, "undershoot_penalty": round(undershoot_penalty,3),
                "d3_score": d3_score}




# ── D5: Trajectory Efficiency ────────────────────────────────────

class D5_TrajectoryEfficiency:
    name = "D5_trajectory_efficiency"
    BASELINE_TOKENS_PER_STEP = 400
    def score(self, trajectory_summary, answer_correct, degenerate=False):
        if degenerate:
            return {"total_steps": 0, "redundancy_rate": 0,
                    "tokens_per_step": 0, "loop_detected": False,
                    "loop_penalty_applied": 0, "efficiency_score": 0.0,
                    "total_tokens": 0, "tokens_per_correct_answer": None,
                    "degenerate": True}

        total_steps    = trajectory_summary.get("total_steps", 0)
        redundancy     = trajectory_summary.get("redundancy_rate", 0)
        tokens_per_step= trajectory_summary.get("tokens_per_step", 0)
        loop_detected  = trajectory_summary.get("loop_detected", False)
        total_tokens   = trajectory_summary.get("total_tokens", 0)
        token_penalty  = min(1.0, tokens_per_step / (self.BASELINE_TOKENS_PER_STEP * 2))
        loop_penalty   = 0.3 if loop_detected else 0.0
        raw  = 1.0 - (redundancy*0.5) - (token_penalty*0.3) - loop_penalty
        eff  = max(0.0, round(raw, 3))
        return {"total_steps": total_steps, "redundancy_rate": redundancy,
                "tokens_per_step": tokens_per_step, "loop_detected": loop_detected,
                "loop_penalty_applied": loop_penalty, "efficiency_score": eff,
                "total_tokens": total_tokens,
                "tokens_per_correct_answer": total_tokens if answer_correct else None}


# ── Quick tests ──────────────────────────────────────────────────

d1 = D1_AnswerCorrectness()
d2 = D2_RetrievalStepQuality()
d3 = D3_PlanningCoherence()
d5 = D5_TrajectoryEfficiency()

# D1 tests
assert d1.score("Bombardier Aerospace made it", "Bombardier Inc.")["near_match"] == True
assert d1.score("Miquette Giraudy is her name", "Miquette Giraudy")["exact_match"] == True
assert d1.score("I don't know", "Paris")["score"] == 0.0
print("D1 tests passed")

# D2 test
fake_steps = [
    {"retrieved_content": ["Steve Hillage is a British musician and spouse of Miquette Giraudy",
                            "The Godfather was directed by Coppola",
                            "Churchill was Prime Minister"]},
]
d2_result = d2.score(fake_steps, ["Steve Hillage is a British musician and spouse of Miquette Giraudy"])
assert d2_result["any_relevant_found"] == True
assert d2_result["avg_precision_at_k"] == round(1/3, 3)
print("D2 tests passed")

# D3 test
traj_good = {"total_steps":2, "unique_queries":["Steve Hillage musician","Steve Hillage spouse"],
             "redundancy_rate":0.0, "loop_detected":False}
traj_bad  = {"total_steps":2, "unique_queries":["Green performer spouse"],
             "redundancy_rate":0.5, "loop_detected":True}
d3_good = d3.score(traj_good, num_hops=2)
d3_bad  = d3.score(traj_bad,  num_hops=2)
assert d3_good["d3_score"] > d3_bad["d3_score"], "Good planning should score higher"
print(f"D3 tests passed — good={d3_good['d3_score']}, bad={d3_bad['d3_score']}")

print("\nAll metric classes ready")

D1 tests passed
D2 tests passed
D3 tests passed — good=0.619, bad=0.0

All metric classes ready


---
## Cell 2 — TrajectoryLogger (stores rewritten_query + retrieval_method)

In [4]:

class TrajectoryLogger:
    def __init__(self, max_steps=10):
        self.max_steps = max_steps
        self.steps = []
        self.loop_detected = False
        self.loop_query = None

    def reset(self):
        self.steps = []
        self.loop_detected = False
        self.loop_query = None

    def log(self, query, retrieved_docs, latency_ms, supporting_paragraphs=None,
            rewritten_query=None, rewrite_rejected=False, retrieval_method=None):
        is_loop = bool(self.steps and self.steps[-1]['query'] == query)
        if is_loop:
            self.loop_detected = True
            self.loop_query = query
        precision_at_k = 0.0
        if supporting_paragraphs and retrieved_docs:
            def _is_relevant(doc):
                doc_words = set(doc.lower().split())
                for sup in supporting_paragraphs:
                    sup_words = set(sup.lower().split())
                    if sup_words and len(sup_words & doc_words) / len(sup_words) > 0.5:
                        return True
                return False
            hits = sum(1 for doc in retrieved_docs if _is_relevant(doc))
            precision_at_k = round(hits / len(retrieved_docs), 3)
        tokens_estimate = (len(query) + sum(len(d) for d in retrieved_docs)) // 4
        step = {
            "step_id":            len(self.steps) + 1,
            "query":              query,
            "rewritten_query":    rewritten_query,
            "rewrite_rejected":   rewrite_rejected,
            "retrieval_method":   retrieval_method,
            "num_docs_retrieved": len(retrieved_docs),
            "retrieved_content":  retrieved_docs,
            "precision_at_k":     precision_at_k,
            "is_loop_step":       is_loop,
            "tokens_estimate":    tokens_estimate,
            "latency_ms":         round(latency_ms, 1)
        }
        self.steps.append(step)
        return step

    def at_limit(self):
        return len(self.steps) >= self.max_steps

    def summary(self):
        if not self.steps: return {}
        n = len(self.steps)
        unique = list(set(s['query'] for s in self.steps))
        redundant = n - len(unique)
        total_tokens = sum(s['tokens_estimate'] for s in self.steps)
        avg_prec = sum(s['precision_at_k'] for s in self.steps) / n
        return {
            "total_steps": n, "unique_queries": unique,
            "redundant_queries": redundant,
            "redundancy_rate": round(redundant / n, 3),
            "loop_detected": self.loop_detected,
            "loop_steps": sum(1 for s in self.steps if s['is_loop_step']),
            "avg_step_precision_at_k": round(avg_prec, 3),
            "total_tokens": total_tokens,
            "total_latency_ms": round(sum(s['latency_ms'] for s in self.steps), 1),
            "tokens_per_step": round(total_tokens / n, 1)
        }
print('TrajectoryLogger ready (with rewrite_rejected + retrieval_method fields)')

TrajectoryLogger ready (with rewrite_rejected + retrieval_method fields)


---
## Cell 3 — Load 50 questions (from Week 3 dataset)

In [5]:
q_path = PROJECT_ROOT / "data/questions/3_musique_50.json"
with open(q_path) as f:
    questions = json.load(f)

print(f"Loaded {len(questions)} questions from {q_path}")
print(f"  Avg KB size per question:    {sum(len(q['paragraphs']) for q in questions)//len(questions)} docs")
print(f"  Avg supporting docs:         {sum(q['num_supporting'] for q in questions)/len(questions):.1f}")
print(f"  Avg distractors:             {sum(q['num_distractors'] for q in questions)/len(questions):.1f}")

Loaded 50 questions from /Users/idhantsingh/Desktop/agenticrag-bench/data/questions/3_musique_50.json
  Avg KB size per question:    20 docs
  Avg supporting docs:         2.0
  Avg distractors:             18.0


---
## Cell 4 — Hybrid retrieval: BM25 + FAISS with reciprocal rank fusion

The core new component. For each query:
1. Get top-k from FAISS (dense vector similarity)
2. Get top-k from BM25 (keyword matching)
3. Combine via Reciprocal Rank Fusion (RRF)
4. Return top-k unique results

RRF formula: `score(doc) = Σ 1/(k + rank(doc))` where k=60 is standard.

In [6]:
class HybridRetriever:
    """Combines FAISS (dense) and BM25 (sparse) retrieval with reciprocal rank fusion."""
    
    RRF_K = 60  # Standard RRF constant
    
    def __init__(self, documents, embedding_model, k=3):
        self.k = k
        self.documents = documents  # list of raw text strings
        
        # Dense: FAISS
        docs = [Document(page_content=p) for p in documents]
        self.faiss_store = FAISS.from_documents(docs, embedding_model)
        
        # Sparse: BM25
        tokenized = [doc.lower().split() for doc in documents]
        self.bm25 = BM25Okapi(tokenized)
    
    def search_faiss_only(self, query):
        """FAISS-only search (Week 4 baseline behavior)."""
        results = self.faiss_store.similarity_search(query, k=self.k)
        return [r.page_content for r in results]
    
    def search_hybrid(self, query):
        """BM25 + FAISS with reciprocal rank fusion."""
        # FAISS results
        faiss_results = self.faiss_store.similarity_search(query, k=self.k)
        faiss_texts = [r.page_content for r in faiss_results]
        
        # BM25 results
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        top_bm25_indices = np.argsort(bm25_scores)[::-1][:self.k]
        bm25_texts = [self.documents[i] for i in top_bm25_indices]
        
        # Reciprocal Rank Fusion
        rrf_scores = {}  # doc_text -> score
        
        for rank, text in enumerate(faiss_texts):
            rrf_scores[text] = rrf_scores.get(text, 0) + 1.0 / (self.RRF_K + rank + 1)
        
        for rank, text in enumerate(bm25_texts):
            rrf_scores[text] = rrf_scores.get(text, 0) + 1.0 / (self.RRF_K + rank + 1)
        
        # Sort by RRF score, return top-k
        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        return [text for text, score in sorted_docs[:self.k]]


# ── Test hybrid retrieval ─────────────────────────────────────────

print("Loading embedding model...")
embedding_model = OllamaEmbeddings(model="nomic-embed-text")
_ = embedding_model.embed_query("test")
print("nomic-embed-text OK")

# Quick test with a known failure case from Week 4
test_q = next(q for q in questions if q['id'] == '2hop__252311_366220')
print(f"\nTest question: {test_q['question']}")
print(f"Expected: {test_q['answer']}")

retriever = HybridRetriever(test_q['paragraphs'], embedding_model, k=3)

# FAISS-only
faiss_only = retriever.search_faiss_only("UHF film distributor")
print(f"\nFAISS-only results:")
for i, doc in enumerate(faiss_only):
    relevant = any(sup[:80].lower() in doc.lower() for sup in test_q['supporting_paragraphs'])
    print(f"  {i+1}. {'✓' if relevant else '✗'} {doc[:80]}...")

# Hybrid
hybrid = retriever.search_hybrid("UHF film distributor")
print(f"\nHybrid (BM25+FAISS) results:")
for i, doc in enumerate(hybrid):
    relevant = any(sup[:80].lower() in doc.lower() for sup in test_q['supporting_paragraphs'])
    print(f"  {i+1}. {'✓' if relevant else '✗'} {doc[:80]}...")

print("\nHybridRetriever ready")

Loading embedding model...
nomic-embed-text OK

Test question: Who founded the company that distributed the film UHF?
Expected: Mike Medavoy

FAISS-only results:
  1. ✗ SModcast Pictures is an American film distribution company and a film and televi...
  2. ✗ Amblin Entertainment is an American film production company founded by director ...
  3. ✓ Morris Mike Medavoy (born January 21, 1941) is an American film producer and exe...

Hybrid (BM25+FAISS) results:
  1. ✗ SModcast Pictures is an American film distribution company and a film and televi...
  2. ✗ Videodrome is a 1983 Canadian science fiction body horror film written and direc...
  3. ✗ Amblin Entertainment is an American film production company founded by director ...

HybridRetriever ready


---
## Cell 5 — Rewrite validation

Detect bad LLM rewrites (meta-language) and fall back to original query.

Bad rewrites from Week 4 analysis:
- `"Similarity search for teams with similar characteristics..."` 
- `"Similar vectors to individuals named Tyler MacDuff"`
- `"Find locations similar to Imperial..."`
- Contains `->` syntax artifacts

In [7]:
# Bad rewrite detection patterns
BAD_REWRITE_PREFIXES = [
    "similarity search",
    "similar vectors",
    "find locations",
    "find songs",
    "find ",  # generic "Find X" pattern
    "search for",
    "vector search",
]

BAD_REWRITE_CONTAINS = [
    "->",        # LLM artifact from few-shot confusion
    "similar character",
    "similar documents",
    "with similar",
]

def validate_rewrite(original_query, rewritten_query):
    """
    Returns the rewritten query if valid, or the original if the rewrite is bad.
    Also returns whether the rewrite was rejected.
    """
    rw_lower = rewritten_query.lower().strip()
    
    # Check prefixes
    for prefix in BAD_REWRITE_PREFIXES:
        if rw_lower.startswith(prefix):
            return original_query, True
    
    # Check contains
    for pattern in BAD_REWRITE_CONTAINS:
        if pattern in rw_lower:
            return original_query, True
    
    # Rewrite is too long (LLM generated a sentence instead of a query)
    if len(rewritten_query) > len(original_query) * 3:
        return original_query, True
    
    # Rewrite is empty or trivial
    if len(rewritten_query.strip()) < 3:
        return original_query, True
    
    return rewritten_query, False


# ── Test rewrite validation ──────────────────────────────────────

test_cases = [
    ("Tyler MacDuff child education", "Similarity search for educational content related to Tyler MacDuff"),
    ("Tyler MacDuff", "Similar vectors to individuals named Tyler MacDuff"),
    ("author of The Red Tree", "Similarity search for literary works written by The Red Tree author"),
    ("What league does Crotone play in?", "Similarity search for teams with similar characteristics"),
    ("author of The Timothy Files", 'author of The Timothy Files\" -> \"Timothy Files author'),
    # Good rewrites that should NOT be rejected:
    ("Caroline LeRoy spouse", "spouse of Caroline LeRoy"),
    ("Who owns Bombardier Aerospace?", "Bombardier Aerospace parent company"),
    ("Kimbrough Memorial Stadium location", "Kimbrough Memorial Stadium Texas"),
]

print("Rewrite validation test results:")
rejected_count = 0
for orig, rewrite in test_cases:
    result, rejected = validate_rewrite(orig, rewrite)
    status = "REJECTED ✗" if rejected else "ACCEPTED ✓"
    rejected_count += int(rejected)
    print(f"  {status}: '{rewrite[:50]}' → using '{result[:50]}'")

assert rejected_count == 5, f"Expected 5 rejections, got {rejected_count}"
print(f"\nRewrite validation ready — {rejected_count}/8 bad rewrites caught")

Rewrite validation test results:
  REJECTED ✗: 'Similarity search for educational content related ' → using 'Tyler MacDuff child education'
  REJECTED ✗: 'Similar vectors to individuals named Tyler MacDuff' → using 'Tyler MacDuff'
  REJECTED ✗: 'Similarity search for literary works written by Th' → using 'author of The Red Tree'
  REJECTED ✗: 'Similarity search for teams with similar character' → using 'What league does Crotone play in?'
  REJECTED ✗: 'author of The Timothy Files" -> "Timothy Files aut' → using 'author of The Timothy Files'
  ACCEPTED ✓: 'spouse of Caroline LeRoy' → using 'spouse of Caroline LeRoy'
  ACCEPTED ✓: 'Bombardier Aerospace parent company' → using 'Bombardier Aerospace parent company'
  ACCEPTED ✓: 'Kimbrough Memorial Stadium Texas' → using 'Kimbrough Memorial Stadium Texas'

Rewrite validation ready — 5/8 bad rewrites caught


---
## Cell 6 — Build agent (3 configurations)

The agent tool is parameterised by:
- `USE_REWRITE_VALIDATION`: whether to validate/reject bad rewrites
- `USE_HYBRID_RETRIEVAL`: whether to use BM25+FAISS or FAISS-only

In [8]:
TRAJECTORY_LOGGER = TrajectoryLogger(max_steps=10)
current_context = {"retriever": None, "supporting_paragraphs": []}

# ── Configuration flags ──────────────────────────────────────────
USE_REWRITE_VALIDATION = False  # Set per-config
USE_HYBRID_RETRIEVAL   = False  # Set per-config

MAX_CHARS_PER_DOC = 500  # Truncate tool output for LLM context

llm = ChatOllama(model="llama3.1:8b", temperature=0)


@tool
def search_knowledge_base(query: str) -> str:
    """
    Search the knowledge base for information relevant to the query.
    For multi-hop questions ALWAYS search at least twice:
    - First: find what the subject IS
    - Second: find the specific fact about that subject
    Never repeat the same query. If results are unhelpful, change the query.
    """
    if TRAJECTORY_LOGGER.at_limit():
        return "[SEARCH LIMIT REACHED: provide your best answer now.]"

    # Query Rewriting: ask the LLM to generate a cleaner search query
    rewrite_prompt = (
        "Rewrite this search query to be effective for a vector similarity search. "
        "Extract key entities and facts. Remove filler words. "
        "Output ONLY the rewritten query, nothing else.\n"
        f"Original query: {query}"
    )
    try:
        rewritten = llm.invoke([("user", rewrite_prompt)]).content.strip().strip('\"\'')
    except Exception:
        rewritten = query

    # Rewrite validation (Lever 2)
    rewrite_rejected = False
    if USE_REWRITE_VALIDATION:
        rewritten, rewrite_rejected = validate_rewrite(query, rewritten)

    # Retrieval (Lever 1)
    t0 = time.time()
    if USE_HYBRID_RETRIEVAL:
        texts = current_context["retriever"].search_hybrid(rewritten)
        retrieval_method = "hybrid_bm25_faiss"
    else:
        texts = current_context["retriever"].search_faiss_only(rewritten)
        retrieval_method = "faiss_only"
    latency = (time.time() - t0) * 1000

    step = TRAJECTORY_LOGGER.log(
        query, texts, latency,
        current_context["supporting_paragraphs"],
        rewritten_query=rewritten,
        rewrite_rejected=rewrite_rejected,
        retrieval_method=retrieval_method
    )

    # Truncate for LLM context — full text is already saved in trajectory
    truncated = [t[:MAX_CHARS_PER_DOC] for t in texts]

    warning = ""
    if step["is_loop_step"]:
        warning = "\n\n[You already searched this query. Use a DIFFERENT query.]"

    # Nudge agent to keep searching after first step
    nudge = ""
    if len(TRAJECTORY_LOGGER.steps) == 1:
        nudge = "\n\n[NOTE: You have completed 1 search. Multi-hop questions require at least 2 searches. Search again with a DIFFERENT query before answering.]"

    return "\n\n---\n\n".join(truncated) + warning + nudge


SYSTEM_PROMPT = """You are a research assistant answering multi-hop questions.
These questions require combining facts from 2+ sources.

RULES:
1. ALWAYS search at least twice before answering.
2. First search: find what the main entity IS.
3. Second search: find the specific fact about that entity.
4. Never repeat a query — vary your searches.
5. Answer concisely after at least 2 searches."""

agent = create_react_agent(llm, [search_knowledge_base], prompt=SYSTEM_PROMPT)

print("Agent ready — k=3, system prompt ON, max_steps=10")
print(f"  Rewrite validation: {USE_REWRITE_VALIDATION}")
print(f"  Hybrid retrieval:   {USE_HYBRID_RETRIEVAL}")

Agent ready — k=3, system prompt ON, max_steps=10
  Rewrite validation: False
  Hybrid retrieval:   False


/var/folders/tl/vfsv8k5j2bv967f2r2x0ph7m0000gn/T/ipykernel_52377/37988185.py:85: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, [search_knowledge_base], prompt=SYSTEM_PROMPT)


---
## Cell 7 — Run function with timeout + recursion_limit

In [9]:
d1_metric = D1_AnswerCorrectness()
d2_metric = D2_RetrievalStepQuality()
d3_metric = D3_PlanningCoherence()
d5_metric = D5_TrajectoryEfficiency()

# Timeout wrapper
class QuestionTimeoutError(Exception):
    pass

def _timeout_handler(signum, frame):
    raise QuestionTimeoutError("Question timed out")

QUESTION_TIMEOUT = 120  # seconds per question


def run_question(q):
    global current_context
    TRAJECTORY_LOGGER.reset()

    # Build per-question hybrid retriever
    retriever = HybridRetriever(q['paragraphs'], embedding_model, k=3)
    current_context["retriever"]              = retriever
    current_context["supporting_paragraphs"]  = q['supporting_paragraphs']

    t0 = time.time()

    # Timeout so one question can't hang forever
    old_handler = signal.signal(signal.SIGALRM, _timeout_handler)
    signal.alarm(QUESTION_TIMEOUT)

    try:
        result  = agent.invoke(
            {"messages": [("user", q["question"])]},
            config={"recursion_limit": 28}
        )
        answer  = result["messages"][-1].content
        success = True
    except QuestionTimeoutError:
        answer  = "TIMEOUT"
        success = False
        print(f"    ⚠ TIMEOUT after {QUESTION_TIMEOUT}s")
    except Exception as e:
        answer  = f"ERROR: {e}"
        success = False
        print(f"    ⚠ ERROR: {e}")
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)

    wall_ms   = (time.time() - t0) * 1000
    traj_sum  = TRAJECTORY_LOGGER.summary()

    d1 = d1_metric.score(answer, q["answer"])
    d2 = d2_metric.score(TRAJECTORY_LOGGER.steps, q["supporting_paragraphs"])
    d3 = d3_metric.score(traj_sum, q["num_hops"])
    d5 = d5_metric.score(traj_sum, d1["exact_match"], degenerate=d1.get("degenerate_output", False))

    return {
        "question_id":        q["id"],
        "question":           q["question"],
        "ground_truth":       q["answer"],
        "num_hops":           q["num_hops"],
        "difficulty":         q["difficulty"],
        "predicted_answer":   answer,
        "success":            success,
        "wall_latency_ms":    round(wall_ms, 1),
        "config": {
            "rewrite_validation": USE_REWRITE_VALIDATION,
            "hybrid_retrieval":   USE_HYBRID_RETRIEVAL,
        },
        "trajectory":         TRAJECTORY_LOGGER.steps.copy(),
        "trajectory_summary": traj_sum,
        "D1_answer_correctness":     d1,
        "D2_retrieval_step_quality": d2,
        "D3_planning_coherence":     d3,
        "D5_trajectory_efficiency":  d5,
    }

print("Run function ready — scores D1, D2, D3, D5 per question")

Run function ready — scores D1, D2, D3, D5 per question


---
## Cell 8 — Config C: Full pipeline (rewrite validation ON + hybrid ON)

This is the main experiment. Configs A and B can be run later for ablation.

Each question takes ~10–50s. Total ~10–15 minutes. Results saved after every question.

In [10]:
# ── CONFIG C: rewrite validation ON + hybrid retrieval ON ────────
USE_REWRITE_VALIDATION = True
USE_HYBRID_RETRIEVAL   = True

config_name = "C_hybrid_validated"
out_path = PROJECT_ROOT / f"data/trajectories/5_agent_results_{config_name}.json"

print(f"CONFIG C — Full pipeline: rewrite validation ON + hybrid BM25+FAISS")
print(f"k=3 | system prompt ON | nomic-embed-text | max_steps=10")
print(f"recursion_limit=28 | timeout=120s per question")
print(f"Output: {out_path}")
print("="*65)
print()

results_c = []
for i, q in enumerate(questions):
    print(f"Q{i+1}/50: {q['question'][:65]}...")
    print(f"  Expected: {q['answer']} | Hops: {q['num_hops']} | RC={q['difficulty']['reasoning_complexity']} RD={q['difficulty']['retrieval_difficulty']}")

    r = run_question(q)
    results_c.append(r)

    d1s = r['D1_answer_correctness']['score']
    d2s = r['D2_retrieval_step_quality']['d2_score']
    d3s = r['D3_planning_coherence']['d3_score']
    d5s = r['D5_trajectory_efficiency']['efficiency_score']
    wall = r['wall_latency_ms'] / 1000

    print(f"  Predicted: {r['predicted_answer'][:70]}")
    print(f"  D1={d1s:.3f}  D2={d2s:.3f}  D3={d3s:.3f}  D5={d5s:.3f}  ({wall:.0f}s)")

    # Show rewrites
    for step in r['trajectory']:
        rw = step.get('rewritten_query', '')
        rejected = step.get('rewrite_rejected', False)
        method = step.get('retrieval_method', 'unknown')
        tag = " [REJECTED]" if rejected else ""
        print(f"    [{method}] '{step['query'][:40]}' → '{rw[:40]}'{tag}")
    print()

    # Save after every question
    with open(out_path, 'w') as f:
        json.dump(results_c, f, indent=2)

print("="*65)
correct_c = sum(1 for r in results_c if r['D1_answer_correctness']['exact_match'])
print(f"Config C complete: {correct_c}/50 correct ({correct_c/50:.0%})")
print(f"Saved to {out_path}")

CONFIG C — Full pipeline: rewrite validation ON + hybrid BM25+FAISS
k=3 | system prompt ON | nomic-embed-text | max_steps=10
recursion_limit=28 | timeout=120s per question
Output: /Users/idhantsingh/Desktop/agenticrag-bench/data/trajectories/5_agent_results_C_hybrid_validated.json

Q1/50: Who is the spouse of the Green performer?...
  Expected: Miquette Giraudy | Hops: 2 | RC=2 RD=3
  Predicted: Based on your searches, I can conclude that Billie Joe Armstrong is th
  D1=0.000  D2=0.467  D3=0.760  D5=0.898  (30s)
    [hybrid_bm25_faiss] 'Who is Billie Joe Armstrong's spouse?' → 'Billie Joe Armstrong spouse'
    [hybrid_bm25_faiss] 'What is the name of the lead singer of t' → 'What is the name of the lead singer of t' [REJECTED]
    [hybrid_bm25_faiss] 'Who is the lead singer of Green Day?' → 'Who is the lead singer of Green Day?' [REJECTED]

Q2/50: Who founded the company that distributed the film UHF?...
  Expected: Mike Medavoy | Hops: 2 | RC=2 RD=3
  Predicted: The company that distr

---
## Cell 9 — Analysis: Week 4 vs Week 5 Config C

In [11]:
# Load Week 4 baseline for comparison
with open(PROJECT_ROOT / "data/trajectories/4_agent_results_clean.json") as f:
    results_w4 = json.load(f)

def summarize_results(results, label):
    n = len(results)
    correct = sum(1 for r in results if r['D1_answer_correctness']['exact_match'])
    degen = sum(1 for r in results if r['D1_answer_correctness'].get('degenerate_output', False))
    nd = [r for r in results if not r['D1_answer_correctness'].get('degenerate_output', False)]
    n_nd = len(nd)
    
    avg_d1 = sum(r['D1_answer_correctness']['score'] for r in results) / n
    avg_d2 = sum(r['D2_retrieval_step_quality']['d2_score'] for r in nd) / n_nd if n_nd else 0
    avg_d3 = sum(r['D3_planning_coherence']['d3_score'] for r in nd) / n_nd if n_nd else 0
    avg_d5 = sum(r['D5_trajectory_efficiency']['efficiency_score'] for r in nd) / n_nd if n_nd else 0
    d2_zero = sum(1 for r in nd if r['D2_retrieval_step_quality']['d2_score'] == 0)
    d2_high = sum(1 for r in nd if r['D2_retrieval_step_quality']['d2_score'] > 0.467)
    loops = sum(1 for r in results if r['trajectory_summary'].get('loop_detected'))
    
    # D2 for correct vs incorrect
    corr = [r for r in nd if r['D1_answer_correctness']['exact_match']]
    incorr = [r for r in nd if not r['D1_answer_correctness']['exact_match']]
    d2_corr = sum(r['D2_retrieval_step_quality']['d2_score'] for r in corr)/len(corr) if corr else 0
    d2_incorr = sum(r['D2_retrieval_step_quality']['d2_score'] for r in incorr)/len(incorr) if incorr else 0
    
    # Rewrite rejections
    total_steps = sum(len(r.get('trajectory', [])) for r in results)
    rejected_steps = sum(
        sum(1 for s in r.get('trajectory', []) if s.get('rewrite_rejected', False))
        for r in results
    )
    
    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"{'='*60}")
    print(f"Accuracy:      {correct}/{n} = {correct/n:.0%}")
    print(f"Avg D1:        {avg_d1:.3f}")
    print(f"Avg D2:        {avg_d2:.3f}   (D2=0: {d2_zero}/{n_nd}  |  D2>0.467: {d2_high}/{n_nd})")
    print(f"Avg D3:        {avg_d3:.3f}")
    print(f"Avg D5:        {avg_d5:.3f}")
    print(f"Loops:         {loops}/{n}")
    print(f"Degenerate:    {degen}/{n}")
    print(f"D2 correct:    {d2_corr:.3f}  |  D2 incorrect: {d2_incorr:.3f}")
    if rejected_steps > 0:
        print(f"Rewrites rejected: {rejected_steps}/{total_steps}")
    
    return {
        'correct': correct, 'n': n, 'd2': avg_d2, 'd2_zero': d2_zero,
        'd2_high': d2_high, 'd3': avg_d3, 'd5': avg_d5
    }

s_w4 = summarize_results(results_w4, "Week 4 — FAISS only + query rewriting (baseline)")
s_c  = summarize_results(results_c,  "Week 5 Config C — Hybrid BM25+FAISS + rewrite validation")

print(f"\n{'='*60}")
print(f"COMPARISON: Week 4 → Week 5 Config C")
print(f"{'='*60}")
print(f"D1 Accuracy:  {s_w4['correct']}/{s_w4['n']} → {s_c['correct']}/{s_c['n']}")
print(f"D2 Retrieval: {s_w4['d2']:.3f} → {s_c['d2']:.3f}  ({(s_c['d2']-s_w4['d2'])/s_w4['d2']*100:+.1f}%)")
print(f"D2=0 count:   {s_w4['d2_zero']} → {s_c['d2_zero']}")
print(f"D2>0.467:     {s_w4['d2_high']} → {s_c['d2_high']}")
print(f"D3 Planning:  {s_w4['d3']:.3f} → {s_c['d3']:.3f}")
print(f"D5 Efficiency:{s_w4['d5']:.3f} → {s_c['d5']:.3f}")


Week 4 — FAISS only + query rewriting (baseline)
Accuracy:      15/50 = 30%
Avg D1:        0.345
Avg D2:        0.302   (D2=0: 15/48  |  D2>0.467: 8/48)
Avg D3:        0.647
Avg D5:        0.860
Loops:         4/50
Degenerate:    2/50
D2 correct:    0.480  |  D2 incorrect: 0.221

Week 5 Config C — Hybrid BM25+FAISS + rewrite validation
Accuracy:      24/50 = 48%
Avg D1:        0.506
Avg D2:        0.416   (D2=0: 7/48  |  D2>0.467: 15/48)
Avg D3:        0.698
Avg D5:        0.883
Loops:         1/50
Degenerate:    2/50
D2 correct:    0.514  |  D2 incorrect: 0.318
Rewrites rejected: 39/101

COMPARISON: Week 4 → Week 5 Config C
D1 Accuracy:  15/50 → 24/50
D2 Retrieval: 0.302 → 0.416  (+37.7%)
D2=0 count:   15 → 7
D2>0.467:     8 → 15
D3 Planning:  0.647 → 0.698
D5 Efficiency:0.860 → 0.883


---
## Cell 10 — Per-question D2 change analysis

For each question, compare D2 from Week 4 (FAISS-only) vs Week 5 Config C (hybrid).

In [12]:
print("PER-QUESTION D2 CHANGE: Week 4 → Week 5 Config C")
print("="*70)

d2_improved = []
d2_degraded = []
d2_same = []
d2_zero_fixed = []  # Was D2=0, now D2>0
newly_correct = []  # Was wrong, now correct
newly_wrong = []    # Was correct, now wrong

for w4, w5 in zip(results_w4, results_c):
    assert w4['question_id'] == w5['question_id']
    d2_w4 = w4['D2_retrieval_step_quality']['d2_score']
    d2_w5 = w5['D2_retrieval_step_quality']['d2_score']
    c_w4 = w4['D1_answer_correctness']['exact_match']
    c_w5 = w5['D1_answer_correctness']['exact_match']
    
    entry = {
        'qid': w4['question_id'],
        'question': w4['question'][:60],
        'd2_w4': d2_w4, 'd2_w5': d2_w5,
        'correct_w4': c_w4, 'correct_w5': c_w5,
    }
    
    if d2_w5 > d2_w4:
        d2_improved.append(entry)
    elif d2_w5 < d2_w4:
        d2_degraded.append(entry)
    else:
        d2_same.append(entry)
    
    if d2_w4 == 0 and d2_w5 > 0:
        d2_zero_fixed.append(entry)
    
    if not c_w4 and c_w5:
        newly_correct.append(entry)
    elif c_w4 and not c_w5:
        newly_wrong.append(entry)

print(f"D2 improved: {len(d2_improved)}  |  D2 degraded: {len(d2_degraded)}  |  D2 same: {len(d2_same)}")
print(f"D2=0 fixed (was zero, now >0): {len(d2_zero_fixed)}")
print(f"Newly correct: {len(newly_correct)}  |  Newly wrong: {len(newly_wrong)}")

print(f"\n--- D2=0 FIXED (the hybrid retrieval wins) ---")
for e in d2_zero_fixed:
    correct_tag = "→ CORRECT" if e['correct_w5'] else ""
    print(f"  {e['qid'][:30]:30}  D2: {e['d2_w4']:.3f} → {e['d2_w5']:.3f}  {correct_tag}")

if newly_correct:
    print(f"\n--- NEWLY CORRECT (wrong in Week 4, correct in Week 5) ---")
    for e in newly_correct:
        print(f"  {e['question']}")
        print(f"    D2: {e['d2_w4']:.3f} → {e['d2_w5']:.3f}")

if newly_wrong:
    print(f"\n--- NEWLY WRONG (correct in Week 4, wrong in Week 5) ---")
    for e in newly_wrong:
        print(f"  {e['question']}")
        print(f"    D2: {e['d2_w4']:.3f} → {e['d2_w5']:.3f}")

# Save per-question comparison
comparison = {
    'd2_improved': d2_improved,
    'd2_degraded': d2_degraded,
    'd2_same': d2_same,
    'd2_zero_fixed': d2_zero_fixed,
    'newly_correct': newly_correct,
    'newly_wrong': newly_wrong,
}
comp_path = PROJECT_ROOT / "data/trajectories/5_w4_vs_w5_comparison.json"
with open(comp_path, 'w') as f:
    json.dump(comparison, f, indent=2)
print(f"\nSaved comparison to {comp_path}")

PER-QUESTION D2 CHANGE: Week 4 → Week 5 Config C
D2 improved: 22  |  D2 degraded: 5  |  D2 same: 23
D2=0 fixed (was zero, now >0): 10
Newly correct: 13  |  Newly wrong: 4

--- D2=0 FIXED (the hybrid retrieval wins) ---
  2hop__166824_185403             D2: 0.000 → 0.333  
  2hop__698577_851686             D2: 0.000 → 0.466  → CORRECT
  2hop__195917_575657             D2: 0.000 → 0.333  → CORRECT
  2hop__505860_650613             D2: 0.000 → 0.333  → CORRECT
  2hop__821197_368148             D2: 0.000 → 0.466  → CORRECT
  2hop__702496_430061             D2: 0.000 → 0.600  → CORRECT
  2hop__647996_551681             D2: 0.000 → 0.466  → CORRECT
  2hop__574419_270105             D2: 0.000 → 0.466  → CORRECT
  2hop__700093_455653             D2: 0.000 → 0.466  
  2hop__765678_310556             D2: 0.000 → 0.466  

--- NEWLY CORRECT (wrong in Week 4, correct in Week 5) ---
  What administrative territorial entity is the owner of Ciuda
    D2: 0.333 → 0.466
  What province shares a border w

---
## Cell 11 — Auto-generate observations

In [13]:
n = len(results_c)
correct = sum(1 for r in results_c if r['D1_answer_correctness']['exact_match'])
degen = sum(1 for r in results_c if r['D1_answer_correctness'].get('degenerate_output', False))
nd = [r for r in results_c if not r['D1_answer_correctness'].get('degenerate_output', False)]
n_nd = len(nd)

avg_d1 = sum(r['D1_answer_correctness']['score'] for r in results_c) / n
avg_d2 = sum(r['D2_retrieval_step_quality']['d2_score'] for r in nd) / n_nd
avg_d3 = sum(r['D3_planning_coherence']['d3_score'] for r in nd) / n_nd
avg_d5 = sum(r['D5_trajectory_efficiency']['efficiency_score'] for r in nd) / n_nd

d2_zero = sum(1 for r in nd if r['D2_retrieval_step_quality']['d2_score'] == 0)
d2_high = sum(1 for r in nd if r['D2_retrieval_step_quality']['d2_score'] > 0.467)
loops = sum(1 for r in results_c if r['trajectory_summary'].get('loop_detected'))

# Avg steps
avg_steps = sum(r['trajectory_summary'].get('total_steps', 0) for r in nd) / n_nd

# Tokens per correct
ct = [r['D5_trajectory_efficiency']['tokens_per_correct_answer'] for r in results_c
      if r['D1_answer_correctness']['exact_match'] 
      and r['D5_trajectory_efficiency'].get('tokens_per_correct_answer') is not None]
avg_tpca = sum(ct)/len(ct) if ct else 0

# Rewrite rejections
total_steps = sum(len(r.get('trajectory', [])) for r in results_c)
rejected_steps = sum(
    sum(1 for s in r.get('trajectory', []) if s.get('rewrite_rejected', False))
    for r in results_c
)

obs = f"""# Week 5 Observations — AgenticRAG-Bench
Date: {time.strftime('%Y-%m-%d')}

## What was built
- Hybrid retrieval: BM25 (keyword) + FAISS (dense) with reciprocal rank fusion (RRF, k=60)
- Rewrite validation: detect and reject bad LLM rewrites (meta-language fallback)
- Ablation-ready notebook: 3 configurations (A=baseline, B=rewrite only, C=full pipeline)

## Config C results (50 questions, rewrite validation ON + hybrid ON)
- Accuracy:       {correct}/{n} = {correct/n:.0%}  (Week 4: 30%)
- Avg D1 score:   {avg_d1:.3f}
- Avg D2 score:   {avg_d2:.3f}  (Week 4: 0.302)
- Avg D3 score:   {avg_d3:.3f}  (Week 4: 0.647)
- Avg D5 score:   {avg_d5:.3f}  (Week 4: 0.860)
- Avg steps/q:    {avg_steps:.2f}
- Loops detected: {loops}/{n}
- Degenerate:     {degen}/{n}
- Tokens/correct: {avg_tpca:.1f}
- D2=0 count:     {d2_zero}/{n_nd}  (Week 4: 15/48)
- D2>0.467 count: {d2_high}/{n_nd}  (Week 4: 8/48)
- Rewrites rejected: {rejected_steps}/{total_steps}

## Per-question changes (Week 4 → Week 5)
- D2 improved: {len(d2_improved)} questions
- D2 degraded: {len(d2_degraded)} questions
- D2 unchanged: {len(d2_same)} questions
- D2=0 fixed (was zero, now >0): {len(d2_zero_fixed)}
- Newly correct: {len(newly_correct)}
- Newly wrong: {len(newly_wrong)}

## Key observations
- TBD after reviewing results

## Week 6 plan
- TBD after reviewing results
"""

obs_path = PROJECT_ROOT / "notes/5_observations.md"
with open(obs_path, 'w') as f:
    f.write(obs)
print(f"Observations saved to {obs_path}")
print("\n" + obs)

Observations saved to /Users/idhantsingh/Desktop/agenticrag-bench/notes/5_observations.md

# Week 5 Observations — AgenticRAG-Bench
Date: 2026-06-15

## What was built
- Hybrid retrieval: BM25 (keyword) + FAISS (dense) with reciprocal rank fusion (RRF, k=60)
- Rewrite validation: detect and reject bad LLM rewrites (meta-language fallback)
- Ablation-ready notebook: 3 configurations (A=baseline, B=rewrite only, C=full pipeline)

## Config C results (50 questions, rewrite validation ON + hybrid ON)
- Accuracy:       24/50 = 48%  (Week 4: 30%)
- Avg D1 score:   0.506
- Avg D2 score:   0.416  (Week 4: 0.302)
- Avg D3 score:   0.698  (Week 4: 0.647)
- Avg D5 score:   0.883  (Week 4: 0.860)
- Avg steps/q:    2.10
- Loops detected: 1/50
- Degenerate:     2/50
- Tokens/correct: 498.5
- D2=0 count:     7/48  (Week 4: 15/48)
- D2>0.467 count: 15/48  (Week 4: 8/48)
- Rewrites rejected: 39/101

## Per-question changes (Week 4 → Week 5)
- D2 improved: 22 questions
- D2 degraded: 5 questions
- D2 un